In [ ]:
# Cell 1 — Setup
from pathlib import Path
import yaml
import pandas as pd
from src.utils import setup_logging, RANDOM_SEED

# Config is loaded from Path.cwd() / "config" / "config.yaml".
# The notebook MUST be run with the project root as the working directory.
# In Jupyter: File > Change Kernel Working Directory, or start jupyter from
# the project root. __file__ is not defined in Jupyter notebooks and cannot
# be used for path resolution.
config_path = Path.cwd() / "config" / "config.yaml"
if not config_path.exists():
    raise FileNotFoundError(
        f"config.yaml not found at {config_path}. "
        "Run Jupyter from the project root directory: "
        "cd /path/to/wikinews-nlp && jupyter notebook"
    )
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

setup_logging(log_file=config["logging"]["log_file"])
random_seed = config.get("random_seed", RANDOM_SEED)

import src.data_loader as data_loader
import src.data_inspector as data_inspector
import src.data_normalizer as data_normalizer
import src.preprocessing as preprocessing
import src.ner as ner
import src.summarizer as summarizer
import src.similarity as similarity
import src.topic_predictor as topic_predictor
from src.utils import release_model

In [ ]:
# Cell 2 — Validate config
summarizer.validate_summarization_config(config)
ner.validate_ner_config(config)
# Both fail fast if config values are internally inconsistent.
# Catches errors before any model is loaded.

In [ ]:
# Cell 3 — Download data
raw_path = data_loader.download_dataset(
    config["data"]["source_url"],
    config["data"]["raw_path"],
)

In [ ]:
# Cell 4 — Raw profile (human review gate)
fmt = data_inspector.detect_format(str(raw_path))
profile = data_inspector.raw_profile(str(raw_path), fmt)
data_inspector.print_raw_profile(profile)
# HUMAN: review output. Confirm format and record count look correct before continuing.

In [ ]:
# Cell 5 — Category profile (config selection gate)
category_df = data_inspector.category_profile(str(raw_path), fmt)
data_inspector.print_category_profile(category_df, top_n=50)
display(category_df.head(50))
# HUMAN: review available category labels and counts.
# If topics.selected or countries.selected in config.yaml need changes:
#   1. Edit config/config.yaml.
#   2. Rerun Cell 1 to reload config, or run:
#        config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
#        random_seed = config.get("random_seed", RANDOM_SEED)
#   3. Rerun Cell 2 validation, then continue from Cell 6.
# No interactive input is requested in the notebook; config remains the source of truth.

In [ ]:
# Cell 6 — Normalise (NER pass: en + de)
ner_articles, dropped_ner = data_normalizer.normalise_articles(
    str(raw_path), fmt,
    languages=config["languages"]["ner"],
    topics=config["topics"]["selected"],
    countries=config["countries"]["selected"],
    max_per_topic=config["topics"]["articles_per_topic_max"],
    min_article_length=config["data"]["min_article_length"],
    random_seed=random_seed,
)
data_normalizer.print_normalisation_summary(ner_articles, dropped_ner)

In [ ]:
# Cell 7 — Normalise (summarisation pass: en only)
summ_articles, dropped_summ = data_normalizer.normalise_articles(
    str(raw_path), fmt,
    languages=config["languages"]["summarization"],
    topics=config["topics"]["selected"],
    countries=config["countries"]["selected"],
    max_per_topic=config["topics"]["articles_per_topic_max"],
    min_article_length=config["data"]["min_article_length"],
    random_seed=random_seed,
)
data_normalizer.print_normalisation_summary(summ_articles, dropped_summ)
# Note: ner_articles and summ_articles are separate processing pools.
# With the same random_seed, English selections are identical when both passes
# draw from the same source pool. Use random_seed + 1 for the second pass only
# if the analysis deliberately needs different English samples.

In [ ]:
# Cell 8 — Validate both article sets (human review gate)
ner_report = data_inspector.validate_normalised(ner_articles, config)
data_inspector.print_validation_report(ner_report)
summ_report = data_inspector.validate_normalised(summ_articles, config)
data_inspector.print_validation_report(summ_report)
# Hard guard — do not rely on the human to notice a failed validation.
# Raise immediately so the notebook cannot silently continue on bad data.
if not ner_report.validation_passed or not summ_report.validation_passed:
    raise RuntimeError(
        "Validation failed — review errors above before continuing. "
        "Fix the config or dataset, then re-run from Cell 6."
    )
# HUMAN: also review warnings above. Warnings do not stop execution but
# may indicate countries/topics with too few articles or high rates of missing dates.

In [ ]:
# Cell 9 — Preprocessing
ner_articles = preprocessing.preprocess_articles(ner_articles, config)
summ_articles = preprocessing.preprocess_articles(summ_articles, config)

In [ ]:
# Cell 10 — NER: English
en_ner_pipeline = ner.load_ner_pipeline(config["models"]["ner_english"], "en")
ner_articles = ner.run_ner(
    ner_articles, en_ner_pipeline, "en",
    chunk_size=config["ner"]["chunk_size"],
    chunk_overlap=config["ner"]["chunk_overlap"],
)
del en_ner_pipeline   # caller must del their reference before release_model()
release_model()

In [ ]:
# Cell 11 — NER: German
de_ner_pipeline = ner.load_ner_pipeline(config["models"]["ner_german"], "de")
ner_articles = ner.run_ner(
    ner_articles, de_ner_pipeline, "de",
    chunk_size=config["ner"]["chunk_size"],
    chunk_overlap=config["ner"]["chunk_overlap"],
)
del de_ner_pipeline
release_model()

In [ ]:
# Cell 12 — NER analysis
entity_df = ner.build_entity_dataframe(ner_articles)

for country in sorted(entity_df["country"].dropna().unique()):
    ner.plot_top_entities(entity_df, top_n=20, language="en", country=country)
    ner.plot_top_entities(entity_df, top_n=20, language="de", country=country)

# Programmatically select top 5 English entities per country for dynamics plots
entity_counts = (
    entity_df[entity_df["language"] == "en"]
    .groupby(["country", "entity_text"])["article_id"].nunique()
    .reset_index()
)
for country, group in entity_counts.groupby("country"):
    top_entities = (
        group.sort_values(["article_id", "entity_text"], ascending=[False, True])
        .head(5)["entity_text"]
        .tolist()
    )
    ner.plot_entity_dynamics(
        entity_df,
        entity_names=top_entities,
        language="en",
        country=country,
    )

error_frames = []
for country in sorted(entity_df["country"].dropna().unique()):
    error_frames.append(
        ner.investigate_ner_errors(
            ner_articles,
            language="de",
            error_score_threshold=config["ner"]["error_score_threshold"],
            country=country,
        )
    )
error_df = pd.concat(error_frames, ignore_index=True) if error_frames else pd.DataFrame()
display(error_df.head(20))
print("Note: these are confidence-based candidates for manual review, not confirmed errors.")

In [ ]:
# Cell 13 — Summarisation
summ_pipeline = summarizer.load_summarization_pipeline(config["models"]["summarization"])
summ_articles = summarizer.summarize_articles(summ_articles, summ_pipeline, config)
del summ_pipeline
release_model()
summary_quality_df = summarizer.build_summary_quality_dataframe(summ_articles)
display(summary_quality_df.sort_values("issue_count", ascending=False).head(10))
print("Note: grammar/style findings are based on lightweight surface heuristics, "
      "not a full grammar checker.")

In [ ]:
# Cell 14 — Similarity scoring
embedding_model = similarity.load_embedding_model(config["models"]["similarity"])
summ_articles = similarity.score_all_articles(summ_articles, embedding_model)
del embedding_model
release_model()
print("Note: similarity scores reflect only the first ~256 tokens of each article "
      "due to the embedding model's input limit. See known limitations in SPEC.md.")

In [ ]:
# Cell 15 — Similarity analysis
sim_df = similarity.build_similarity_dataframe(summ_articles)
similarity.plot_similarity_distribution(sim_df, threshold=config["similarity"]["threshold"])
extremes = similarity.explain_similarity_extremes(sim_df)
print("Highest scoring articles (summaries most faithful to original):")
display(pd.DataFrame(extremes["highest"]))
print("Lowest scoring articles (summaries lost most information):")
display(pd.DataFrame(extremes["lowest"]))

In [ ]:
# Cell 16 — Topic prediction
topic_pipeline = topic_predictor.load_topic_pipeline(config["models"]["topic_prediction"])
sampled = topic_predictor.predict_all_topics(
    summ_articles,
    candidate_labels=config["topics"]["selected"],
    topic_pipeline=topic_pipeline,
    hypothesis_template=config["topic_prediction"]["hypothesis_template"],
    sample_size=config["topic_prediction"]["sample_size"],
    random_seed=random_seed,
)
del topic_pipeline
release_model()
eval_results = topic_predictor.evaluate_topic_predictions(sampled)
print(f"Topic prediction accuracy: {eval_results['accuracy']:.1%} "
      f"({eval_results['correct']}/{eval_results['evaluated']} articles evaluated, "
      f"sample size={eval_results['total_sampled']})")
print("Note: accuracy is based on a small sample and is indicative only.")
display(pd.DataFrame(eval_results["results"]))

## Cell 17 — Summary report

_Placeholder for human-written summary findings: country scope, entity counts, most frequent entities, similarity score distributions, topic prediction accuracy, any notable NER errors._